Explore environment settings and catalog table names with isolated, local examples. The examples do not contact Databricks.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.app.config import load_environment, env_value, env_boolean, local_output_root
from pair_recon.core.metadata.constant import CatalogTables, catalog_tables, validate_catalog_schema

show(load_environment, env_value, env_boolean, local_output_root, validate_catalog_schema, catalog_tables)
show_methods(CatalogTables, 'table', 'pair_meta', 'column_map', 'run_status', 'run_error')


## `load_environment`

Load a temporary `.pair_recon.env`. Existing process variables take precedence; the environment is restored after the example.


In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

with TemporaryDirectory() as directory:
    root = Path(directory)
    (root / ".pair_recon.env").write_text("PAIR_RECON_TOY_VALUE=from_file\n")
    with patch("pair_recon.app.config.find_dotenv", return_value=""), patch.dict(os.environ, {}, clear=True):
        print("loaded file:", load_environment(root).name)
        print("value:", os.environ["PAIR_RECON_TOY_VALUE"])
# loaded file: .pair_recon.env
# value: from_file


## `env_value` and `env_boolean`

The first non-empty variable wins. Boolean values accept common true/false spellings.


In [ ]:
with patch.dict(os.environ, {"TOY_PRIMARY": "  ", "TOY_FALLBACK": " hello ", "TOY_ENABLED": "yes"}, clear=True):
    print(env_value("TOY_PRIMARY", "TOY_FALLBACK", default="missing"))
    print(env_value("TOY_UNKNOWN", default="missing"))
    print(env_boolean("TOY_ENABLED"))
    print(env_boolean("TOY_UNKNOWN", default=False))
# hello
# missing
# True
# False


## `local_output_root`

The environment variable takes precedence over the default work directory. The selected directory is created.


In [ ]:
with TemporaryDirectory() as directory:
    target = Path(directory) / "toy-output"
    with patch.dict(os.environ, {"PAIR_RECON_OUTPUT_DIR": str(target)}, clear=True):
        result = local_output_root({"work_dir": "unused"})
        print("selected:", result.name)
        print("created:", result.is_dir())
# selected: toy-output
# created: True


## `validate_catalog_schema` and `catalog_tables`

A valid namespace contains exactly a catalog and schema. Invalid inputs raise `ValueError`.


In [ ]:
print(validate_catalog_schema("  demo.recon  "))
tables = catalog_tables("demo.recon")
print(type(tables).__name__, tables.catalog_schema)
try:
    validate_catalog_schema("demo")
except ValueError as error:
    print(type(error).__name__, str(error))
# demo.recon
# CatalogTables demo.recon
# ValueError catalog schema must be catalog.schema


## `CatalogTables.table` and table properties

The method accepts known PairRecon table names. The properties return fully qualified names.


In [ ]:
print("table:", tables.table("pair_recon_pair_meta"))
print("pair_meta:", tables.pair_meta)
print("column_map:", tables.column_map)
print("run_status:", tables.run_status)
print("run_error:", tables.run_error)
try:
    tables.table("unknown")
except ValueError as error:
    print(type(error).__name__, str(error))
# table: demo.recon.pair_recon_pair_meta
# pair_meta: demo.recon.pair_recon_pair_meta
# column_map: demo.recon.pair_recon_column_map
# run_status: demo.recon.pair_recon_run_status
# run_error: demo.recon.pair_recon_run_error
# ValueError unknown PairRecon table: unknown
